# Week 11: Clustering Part 2
## DBSCAN and Hierarchical Agglomerative Clustering

**Student:** Javed Aamer

Week 10 used k-means to segment customers by engagement and transaction behavior. This week I keep the same customer population and ask a different question:

> **Do density-based and hierarchical methods reveal customer structure that k-means may miss?**

The analysis focuses on DBSCAN and hierarchical agglomerative clustering (HAC). `Churn` is not used to form clusters. I use it only afterward to see whether the discovered groups differ in retention risk.

This keeps the clustering unsupervised while still connecting the results to the broader customer-journey problem.

## Focus for Week 11

The analysis covers:

- DBSCAN and density-based clustering;
- the effect of \(\epsilon\) and MinPts;
- core, border, and noise points;
- non-spherical clusters and outlier handling;
- hierarchical agglomerative clustering;
- single, complete, average, and Ward linkage;
- dendrograms;
- choosing a cut height / number of retained clusters;
- comparing cluster structure rather than assuming every method should agree.

The same standardized numeric customer features from Week 10 are used so distance-based comparisons remain meaningful.

## 1. Questions guiding the analysis

1. How sensitive is DBSCAN to \(\epsilon\) and MinPts?
2. Can DBSCAN identify meaningful dense customer groups while treating unusual observations as noise?
3. Which customers become core, border, or noise points?
4. How do single, complete, average, and Ward linkage behave on the same customer sample?
5. Can a high Silhouette score be misleading if a linkage method isolates only one or two observations?
6. Does Ward linkage recover structure similar to the four-segment k-means solution from Week 10?
7. Which clustering method gives the most useful interpretation for the customer-retention problem?

In [1]:
# Imports and reproducibility
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import linkage, dendrogram

RANDOM_STATE = 42

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [2]:
# Load the same cleaned customer-level dataset used in Week 10
candidate_paths = [
    Path("data/churn_clean.csv"),
    Path("churn_clean.csv"),
]

DATA_PATH = next((p for p in candidate_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find churn_clean.csv. Put it in a data/ folder next to this notebook."
    )

churn = pd.read_csv(DATA_PATH)

print(f"Loaded: {DATA_PATH}")
print(f"Shape: {churn.shape[0]:,} rows x {churn.shape[1]} columns")
print(f"Unique customers: {churn['CustomerID'].nunique():,}")
print(f"Overall churn rate: {churn['Churn'].mean():.2%}")

display(churn.head())

Loaded: data/churn_clean.csv
Shape: 5,630 rows x 20 columns
Unique customers: 5,630
Overall churn rate: 16.84%


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0000,Mobile Phone,3,6.0000,Debit Card,Female,3.0000,3,Laptop & Accessory,2,Single,9,1,11.0000,1.0000,1.0000,5.0000,159.9300
1,50002,1,9.0000,Phone,1,8.0000,UPI,Male,3.0000,4,Mobile,3,Single,7,1,15.0000,0.0000,1.0000,0.0000,120.9000
2,50003,1,9.0000,Phone,1,30.0000,Debit Card,Male,2.0000,4,Mobile,3,Single,6,1,14.0000,0.0000,1.0000,3.0000,120.2800
3,50004,1,0.0000,Phone,3,15.0000,Debit Card,Male,2.0000,4,Laptop & Accessory,5,Single,8,0,23.0000,0.0000,1.0000,3.0000,134.0700
4,50005,1,0.0000,Phone,1,12.0000,CC,Male,3.0000,3,Mobile,5,Single,3,0,11.0000,1.0000,1.0000,3.0000,129.6000


## 2. Variables used for clustering

I retain the seven numeric features used for the Week 10 segmentation:

- `Tenure`
- `SatisfactionScore`
- `Complain`
- `CouponUsed`
- `OrderCount`
- `DaySinceLastOrder`
- `CashbackAmount`

These variables represent relationship length, satisfaction, service problems, incentive use, ordering activity, recency, and cashback behavior.

`CustomerID` is an identifier and is excluded. `Churn` is held out of the clustering inputs so it can be used later as an external profile variable rather than something that directly creates the groups.

In [3]:
cluster_features = [
    "Tenure",
    "SatisfactionScore",
    "Complain",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
    "CashbackAmount",
]

X_raw = churn[cluster_features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

summary = X_raw.describe().T[
    ["mean", "std", "min", "25%", "50%", "75%", "max"]
]
display(summary)

,mean,std,min,25%,50%,75%,max
Tenure,10.1341,8.3580,0.0000,3.0000,9.0000,15.0000,61.0000
SatisfactionScore,3.0668,1.3802,1.0000,2.0000,3.0000,4.0000,5.0000
Complain,0.2849,0.4514,0.0000,0.0000,0.0000,1.0000,1.0000
CouponUsed,1.7169,1.8576,0.0000,1.0000,1.0000,2.0000,16.0000
OrderCount,2.9618,2.8792,1.0000,1.0000,2.0000,3.0000,16.0000
DaySinceLastOrder,4.4593,3.5706,0.0000,2.0000,3.0000,7.0000,46.0000
CashbackAmount,177.2230,49.2070,0.0000,145.7700,163.2800,196.3925,324.9900


## 3. Why I keep the Week 10 scaling decision

DBSCAN and HAC are also distance based. If the raw units are used, a variable with a much larger numeric spread can dominate the geometry.

Week 10 showed that `CashbackAmount` accounted for almost all of the raw variance across these features. I therefore standardize the clustering inputs before applying DBSCAN or HAC.

This also makes \(\epsilon\) easier to interpret as a radius in standardized feature space rather than a mixture of unrelated units.

In [4]:
raw_variance = X_raw.var(ddof=0)
variance_share = (
    raw_variance / raw_variance.sum()
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(
    variance_share.index[::-1],
    variance_share.values[::-1] * 100,
)
ax.set_xlabel("Share of total raw variance (%)")
ax.set_ylabel("Feature")
ax.set_title("Raw feature scales are too different for distance-based clustering")
ax.grid(axis="x")
plt.tight_layout()
plt.show()

display(
    pd.DataFrame({
        "Raw variance": raw_variance,
        "Share of total raw variance": raw_variance / raw_variance.sum(),
    }).sort_values("Share of total raw variance", ascending=False)
)

<Figure size 900x500 with 1 Axes>

,Raw variance,Share of total raw variance
CashbackAmount,"2,420.9023",0.9617
Tenure,69.8429,0.0277
DaySinceLastOrder,12.7471,0.0051
OrderCount,8.2886,0.0033
CouponUsed,3.4502,0.0014
SatisfactionScore,1.9046,0.0008
Complain,0.2037,0.0001


# Part A: DBSCAN

## 4. Choosing a starting MinPts and examining neighborhood distances

DBSCAN has two main controls:

- **\(\epsilon\)** sets the neighborhood radius;
- **MinPts** sets how many observations must fall within that radius for a point to qualify as a core point.

There are seven clustering dimensions here. I use **MinPts = 14** as a practical starting point and then test nearby values rather than assuming one setting is automatically correct.

The k-distance plot below shows the distance from each customer to its 14th nearest neighbor. A bend in this curve can provide a useful starting region for \(\epsilon\).

In [5]:
min_samples = 14

neighbors = NearestNeighbors(n_neighbors=min_samples)
neighbors.fit(X_scaled)
distances, _ = neighbors.kneighbors(X_scaled)

k_distance = np.sort(distances[:, -1])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(np.arange(1, len(k_distance) + 1), k_distance)
ax.set_xlabel("Customers sorted by 14th-neighbor distance")
ax.set_ylabel("Distance to 14th nearest neighbor")
ax.set_title("DBSCAN k-distance curve")
ax.grid()
plt.tight_layout()
plt.show()

quantiles = pd.DataFrame({
    "Quantile": [0.70, 0.75, 0.80, 0.85, 0.90, 0.95],
    "14th-neighbor distance": [
        np.quantile(k_distance, q)
        for q in [0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
    ],
})
display(quantiles)

<Figure size 900x500 with 1 Axes>

,Quantile,14th-neighbor distance
0,0.7000,1.0016
1,0.7500,1.0829
2,0.8000,1.1891
3,0.8500,1.3193
4,0.9000,1.5011
5,0.9500,1.8593


The curve begins to steepen around the 80th to 85th percentile, where the neighborhood distance is roughly 1.2 to 1.3 standardized units. I use that region as a starting point and examine a broader \(\epsilon\) range before choosing the final setting.

## 5. How \(\epsilon\) changes clusters and noise

A small \(\epsilon\) creates very tight neighborhoods. That can fragment the data into many small clusters and label a large share of customers as noise.

A larger \(\epsilon\) connects more observations. If it becomes too large, previously separate dense regions merge.

I hold MinPts at 14 and track:

- number of non-noise clusters;
- fraction of observations labeled as noise;
- Silhouette score among the clustered observations.

In [6]:
eps_values = np.round(np.arange(0.5, 2.21, 0.1), 2)

eps_rows = []

for eps in eps_values:
    model = DBSCAN(
        eps=eps,
        min_samples=min_samples,
    )
    labels = model.fit_predict(X_scaled)

    n_clusters = (
        len(set(labels))
        - (1 if -1 in labels else 0)
    )
    noise_rate = np.mean(labels == -1)

    mask = labels != -1
    sil = np.nan
    if n_clusters >= 2 and mask.sum() > n_clusters:
        sil = silhouette_score(
            X_scaled[mask],
            labels[mask],
        )

    eps_rows.append({
        "Epsilon": eps,
        "Clusters": n_clusters,
        "Noise rate": noise_rate,
        "Silhouette (non-noise)": sil,
    })

eps_results = pd.DataFrame(eps_rows)
display(eps_results)

,Epsilon,Clusters,Noise rate,Silhouette (non-noise)
0,0.5000,27,0.7206,0.1354
1,0.6000,13,0.5705,0.1091
2,0.7000,13,0.4425,0.0960
3,0.8000,6,0.3391,0.2592
4,0.9000,4,0.2591,0.1886
5,1.0000,6,0.1991,0.1912
6,1.1000,7,0.1368,0.1756
7,1.2000,3,0.1014,0.2146
8,1.3000,2,0.0755,0.2333
9,1.4000,2,0.0588,0.2280


In [7]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(
    eps_results["Epsilon"],
    eps_results["Clusters"],
    marker="o",
)
ax.set_xlabel("Epsilon")
ax.set_ylabel("Number of non-noise clusters")
ax.set_title("Small epsilon fragments the data into many dense regions")
ax.grid()
plt.tight_layout()
plt.show()

<Figure size 900x500 with 1 Axes>

In [8]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(
    eps_results["Epsilon"],
    eps_results["Noise rate"] * 100,
    marker="o",
)
ax.set_xlabel("Epsilon")
ax.set_ylabel("Customers labeled as noise (%)")
ax.set_title("Noise falls quickly as epsilon increases")
ax.grid()
plt.tight_layout()
plt.show()

<Figure size 900x500 with 1 Axes>

In [9]:
plot_eps = eps_results.dropna(
    subset=["Silhouette (non-noise)"]
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(
    plot_eps["Epsilon"],
    plot_eps["Silhouette (non-noise)"],
    marker="o",
)
ax.set_xlabel("Epsilon")
ax.set_ylabel("Silhouette score among non-noise customers")
ax.set_title("DBSCAN separation also changes with epsilon")
ax.grid()
plt.tight_layout()
plt.show()

<Figure size 900x500 with 1 Axes>

### Choosing the final DBSCAN setting

A very small radius gives a higher number of clusters but also treats a large portion of the customer base as noise. At the other extreme, a large radius collapses the structure into only a few broad regions.

I select **\(\epsilon = 1.2\)** with **MinPts = 14** as a compromise:

- the radius is close to the bend suggested by the 14th-neighbor distance curve;
- only about 10% of customers are labeled as noise;
- three non-noise clusters remain;
- the non-noise Silhouette score remains interpretable.

This is not presented as a universally optimal DBSCAN setting. It is the setting that gives the most useful balance for this customer data.

## 6. Sensitivity to MinPts

I also hold \(\epsilon = 1.2\) fixed and change MinPts. This shows that DBSCAN's answer is not determined by \(\epsilon\) alone.

In [10]:
minpts_rows = []

for m in [7, 10, 14, 20, 30]:
    model = DBSCAN(
        eps=1.2,
        min_samples=m,
    )
    labels = model.fit_predict(X_scaled)

    n_clusters = (
        len(set(labels))
        - (1 if -1 in labels else 0)
    )
    noise_rate = np.mean(labels == -1)

    mask = labels != -1
    sil = np.nan
    if n_clusters >= 2 and mask.sum() > n_clusters:
        sil = silhouette_score(
            X_scaled[mask],
            labels[mask],
        )

    minpts_rows.append({
        "MinPts": m,
        "Clusters": n_clusters,
        "Noise rate": noise_rate,
        "Silhouette (non-noise)": sil,
    })

minpts_results = pd.DataFrame(minpts_rows)
display(minpts_results)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    minpts_results["MinPts"],
    minpts_results["Noise rate"] * 100,
    marker="o",
)
ax.set_xlabel("MinPts")
ax.set_ylabel("Customers labeled as noise (%)")
ax.set_title("Higher MinPts requires denser neighborhoods")
ax.grid()
plt.tight_layout()
plt.show()

,MinPts,Clusters,Noise rate,Silhouette (non-noise)
0,7,5,0.0650,0.1934
1,10,4,0.0801,0.2080
2,14,3,0.1014,0.2146
3,20,4,0.1405,0.2045
4,30,3,0.1774,0.2227


<Figure size 800x500 with 1 Axes>

Increasing MinPts makes the density requirement stricter, so the noise rate generally rises. The number of clusters can also change because some previously connected regions no longer contain enough core points.

That sensitivity is important: DBSCAN should not be interpreted as producing one uniquely correct segmentation independent of its parameters.

## 7. Final DBSCAN model: core, border, and noise points

I now fit the selected configuration:

- \(\epsilon = 1.2\)
- MinPts = 14

A DBSCAN observation can be:

- **core**: dense enough to anchor a cluster;
- **border**: assigned to a cluster because it is near a core point but is not dense enough itself;
- **noise**: not assigned to any dense region.

In [11]:
dbscan = DBSCAN(
    eps=1.2,
    min_samples=14,
)
db_labels = dbscan.fit_predict(X_scaled)

core_mask = np.zeros(len(churn), dtype=bool)
core_mask[dbscan.core_sample_indices_] = True

point_type = np.where(
    db_labels == -1,
    "Noise",
    np.where(core_mask, "Core", "Border"),
)

dbscan_result = churn.copy()
dbscan_result["DBSCAN_Cluster"] = db_labels
dbscan_result["PointType"] = point_type

point_counts = (
    dbscan_result["PointType"]
    .value_counts()
    .reindex(["Core", "Border", "Noise"])
)

display(point_counts.to_frame("Customers"))

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(
    point_counts.index,
    point_counts.values,
)
ax.set_xlabel("DBSCAN point type")
ax.set_ylabel("Customers")
ax.set_title("Most customers are core points in the selected density structure")
ax.grid(axis="y")
plt.tight_layout()
plt.show()

,Customers
PointType,
Core,4532
Border,527
Noise,571


<Figure size 700x500 with 1 Axes>

## 8. Visualizing core, border, and noise customers

The original space has seven dimensions, so I use PCA only for a two-dimensional visualization. PCA does not determine the DBSCAN labels; it simply provides a view of the standardized feature space.

In [12]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plot_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "PointType": point_type,
    "Cluster": db_labels,
})

fig, ax = plt.subplots(figsize=(9, 6))

for point_kind in ["Core", "Border", "Noise"]:
    subset = plot_df[
        plot_df["PointType"] == point_kind
    ]
    ax.scatter(
        subset["PC1"],
        subset["PC2"],
        s=18,
        alpha=0.55,
        label=point_kind,
    )

ax.set_xlabel("Principal component 1")
ax.set_ylabel("Principal component 2")
ax.set_title("DBSCAN core, border, and noise points in a PCA view")
ax.legend()
ax.grid()
plt.tight_layout()
plt.show()

print(
    "Variance explained by first two PCs: "
    f"{pca.explained_variance_ratio_.sum():.2%}"
)

<Figure size 900x600 with 1 Axes>

Variance explained by first two PCs: 49.26%


The noise observations tend to sit farther from the densest central regions, but the PCA view also shows substantial overlap. That is expected because only part of the seven-dimensional structure is visible in two dimensions.

## 9. Profiling the DBSCAN groups

I profile the three non-noise clusters and the noise observations separately. `Churn` remains an external profile variable and was not used to create these labels.

In [13]:
db_profile = dbscan_result.groupby(
    "DBSCAN_Cluster"
).agg(
    Size=("CustomerID", "size"),
    ChurnRate=("Churn", "mean"),
    Tenure=("Tenure", "mean"),
    Satisfaction=("SatisfactionScore", "mean"),
    Complain=("Complain", "mean"),
    CouponUsed=("CouponUsed", "mean"),
    OrderCount=("OrderCount", "mean"),
    DaySinceLastOrder=("DaySinceLastOrder", "mean"),
    CashbackAmount=("CashbackAmount", "mean"),
)

db_profile["Group"] = [
    "Noise" if idx == -1 else f"Cluster {idx}"
    for idx in db_profile.index
]

display(
    db_profile[
        [
            "Group",
            "Size",
            "ChurnRate",
            "Tenure",
            "Satisfaction",
            "Complain",
            "CouponUsed",
            "OrderCount",
            "DaySinceLastOrder",
            "CashbackAmount",
        ]
    ]
)

,Group,Size,ChurnRate,Tenure,Satisfaction,Complain,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
DBSCAN_Cluster,,,,,,,,,,
-1,Noise,571,0.1366,16.0035,3.2049,0.3853,4.3345,8.4501,8.4343,224.9138
0,Cluster 0,1384,0.3353,9.0137,2.9574,1.0000,1.3338,2.1178,3.6409,170.7984
1,Cluster 1,3654,0.1111,9.5476,3.0818,0.0000,1.4554,2.3914,4.1270,171.5639
2,Cluster 2,21,0.0000,26.4286,3.9048,0.0000,1.2857,8.6190,8.1429,288.5938


In [14]:
churn_db = (
    db_profile[["Group", "ChurnRate", "Size"]]
    .sort_values("ChurnRate")
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(
    churn_db["Group"],
    churn_db["ChurnRate"] * 100,
)
ax.set_xlabel("Observed churn rate (%)")
ax.set_ylabel("DBSCAN group")
ax.set_title("Churn differs across density-based groups")
ax.grid(axis="x")
plt.tight_layout()
plt.show()

<Figure size 800x500 with 1 Axes>

### What DBSCAN found

The largest non-noise groups largely separate customers with and without complaints. The complaint-heavy dense group has a much higher churn rate than the non-complaining dense group.

The noise group is also informative. These customers have substantially higher order counts, coupon use, recency values, and cashback amounts than the main dense groups. DBSCAN therefore treats many high-activity or unusual-behavior customers as outside the dominant density structure instead of forcing them into a cluster.

That is a useful difference from k-means: the unusual observations can remain explicitly unassigned.

# Part B: Hierarchical Agglomerative Clustering

## 10. Why use a sample for HAC?

HAC repeatedly evaluates relationships between clusters and becomes much more computationally demanding as the number of observations grows.

For the linkage comparison and dendrograms, I use a fixed random sample of **1,200 customers** from the same standardized feature matrix. This preserves a substantial cross-section of the customer base while keeping the hierarchical structure readable and reproducible.

In [15]:
rng = np.random.RandomState(RANDOM_STATE)
hac_index = rng.choice(
    len(churn),
    size=1200,
    replace=False,
)

X_hac = X_scaled[hac_index]
hac_customers = churn.iloc[hac_index].copy()

print(f"HAC sample size: {len(hac_customers):,}")
print(
    "Sample churn rate: "
    f"{hac_customers['Churn'].mean():.2%}"
)

HAC sample size: 1,200
Sample churn rate: 16.25%


## 11. Comparing linkage methods

The four linkage rules answer different questions:

- **single linkage:** closest pair of points between clusters;
- **complete linkage:** farthest pair;
- **average linkage:** average pairwise distance;
- **Ward linkage:** merge that produces the smallest increase in total within-cluster squared error.

I compare each method for \(k=2\) through \(k=6\) using Silhouette score.

In [16]:
linkage_methods = [
    "single",
    "complete",
    "average",
    "ward",
]

hac_rows = []
hac_labels = {}

for method in linkage_methods:
    for k in range(2, 7):
        model = AgglomerativeClustering(
            n_clusters=k,
            linkage=method,
        )
        labels = model.fit_predict(X_hac)

        hac_rows.append({
            "Linkage": method,
            "Clusters": k,
            "Silhouette": silhouette_score(
                X_hac,
                labels,
            ),
        })
        hac_labels[(method, k)] = labels

hac_results = pd.DataFrame(hac_rows)
display(
    hac_results.sort_values(
        "Silhouette",
        ascending=False,
    )
)

,Linkage,Clusters,Silhouette
0,single,2,0.5633
10,average,2,0.5633
1,single,3,0.5435
3,single,5,0.5050
2,single,4,0.5044
4,single,6,0.4834
11,average,3,0.4738
5,complete,2,0.4719
6,complete,3,0.4683
12,average,4,0.4667


In [17]:
fig, ax = plt.subplots(figsize=(9, 5))

for method in linkage_methods:
    subset = hac_results[
        hac_results["Linkage"] == method
    ]
    ax.plot(
        subset["Clusters"],
        subset["Silhouette"],
        marker="o",
        label=method,
    )

ax.set_xlabel("Number of retained clusters")
ax.set_ylabel("Silhouette score")
ax.set_title("Linkage choice changes the apparent quality of HAC clusters")
ax.legend()
ax.grid()
plt.tight_layout()
plt.show()

<Figure size 900x500 with 1 Axes>

### A high Silhouette score can still be misleading

Single and average linkage produce the highest Silhouette score at \(k=2\). Before calling that the best segmentation, I inspect the cluster sizes.

A clustering result that isolates one unusual observation from everyone else may score well geometrically but offer very little value as a customer segmentation.

In [18]:
size_rows = []

for method in linkage_methods:
    labels = hac_labels[(method, 2)]
    counts = pd.Series(labels).value_counts().sort_index()

    size_rows.append({
        "Linkage": method,
        "Cluster 0": counts.get(0, 0),
        "Cluster 1": counts.get(1, 0),
        "Smallest cluster": counts.min(),
        "Largest cluster": counts.max(),
    })

hac_size_k2 = pd.DataFrame(size_rows)
display(hac_size_k2)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(
    hac_size_k2["Linkage"],
    hac_size_k2["Smallest cluster"],
)
ax.set_xlabel("Linkage method")
ax.set_ylabel("Customers in smallest k=2 cluster")
ax.set_title("Some high-Silhouette HAC solutions isolate only a few customers")
ax.grid(axis="y")
plt.tight_layout()
plt.show()

,Linkage,Cluster 0,Cluster 1,Smallest cluster,Largest cluster
0,single,1199,1,1,1199
1,complete,1172,28,28,1172
2,average,1199,1,1,1199
3,ward,999,201,201,999


<Figure size 800x500 with 1 Axes>

Single and average linkage each separate essentially one extreme observation from the other 1,199 customers at \(k=2\). Their high Silhouette score therefore reflects outlier separation more than a useful two-segment customer structure.

This is a good example of why a clustering metric should be interpreted together with cluster sizes and business meaning.

## 12. Ward linkage as the closest hierarchical comparison to k-means

Ward linkage minimizes the increase in squared error when clusters merge. That objective is closer to the compact-cluster behavior of k-means than single linkage is.

To make the comparison concrete, I retain four Ward clusters and compare them with a four-cluster k-means solution on the same 1,200-customer sample.

In [19]:
ward4 = hac_labels[("ward", 4)]

kmeans4 = KMeans(
    n_clusters=4,
    random_state=RANDOM_STATE,
    n_init=50,
)
kmeans4_labels = kmeans4.fit_predict(X_hac)

ari_ward_kmeans = adjusted_rand_score(
    ward4,
    kmeans4_labels,
)

ward_counts = pd.Series(
    ward4
).value_counts().sort_index()

print(
    "Adjusted Rand Index between Ward k=4 "
    f"and k-means k=4: {ari_ward_kmeans:.4f}"
)
print("\nWard cluster sizes:")
print(ward_counts.to_string())

Adjusted Rand Index between Ward k=4 and k-means k=4: 0.6076

Ward cluster sizes:
0    225
1    480
2    294
3    201


The Ward and k-means four-cluster assignments have moderate agreement rather than being identical. That is expected: both favor compact groups, but they build those groups in very different ways.

The agreement is still much stronger than the agreement between k-means and single or average linkage, which is consistent with the different geometry of those linkage rules.

## 13. Ward cluster profiles and churn

I profile the Ward \(k=4\) sample solution and again use churn only after clustering.

In [20]:
ward_result = hac_customers.copy()
ward_result["WardCluster"] = ward4

ward_profile = ward_result.groupby(
    "WardCluster"
).agg(
    Size=("CustomerID", "size"),
    ChurnRate=("Churn", "mean"),
    Tenure=("Tenure", "mean"),
    Satisfaction=("SatisfactionScore", "mean"),
    Complain=("Complain", "mean"),
    CouponUsed=("CouponUsed", "mean"),
    OrderCount=("OrderCount", "mean"),
    DaySinceLastOrder=("DaySinceLastOrder", "mean"),
    CashbackAmount=("CashbackAmount", "mean"),
)

display(ward_profile)

fig, ax = plt.subplots(figsize=(8, 5))
ward_churn = ward_profile.sort_values(
    "ChurnRate"
)
ax.barh(
    [f"Ward cluster {i}" for i in ward_churn.index],
    ward_churn["ChurnRate"] * 100,
)
ax.set_xlabel("Observed churn rate (%)")
ax.set_ylabel("Ward cluster")
ax.set_title("Ward clusters also separate customers with different churn rates")
ax.grid(axis="x")
plt.tight_layout()
plt.show()

,Size,ChurnRate,Tenure,Satisfaction,Complain,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
WardCluster,,,,,,,,,
0,225,0.0311,16.4444,3.3778,0.0089,1.0933,2.1244,6.3111,203.3749
1,480,0.1396,6.2938,3.0396,0.0000,1.0229,1.6396,2.7083,156.8336
2,294,0.3027,10.3741,3.0204,1.0000,1.2483,2.0442,3.9932,175.9913
3,201,0.1592,11.3582,3.0050,0.1940,4.2040,7.8159,7.7861,200.1909


<Figure size 800x500 with 1 Axes>

## 14. Dendrograms

A dendrogram shows the sequence of hierarchical merges and the heights at which they occur.

For readability, I use a smaller fixed sample of 250 customers. The dendrogram is not used to refit DBSCAN or k-means; it is a visual explanation of how each linkage rule builds hierarchy.

In [21]:
dendro_rng = np.random.RandomState(7)
dendro_index = dendro_rng.choice(
    len(X_scaled),
    size=250,
    replace=False,
)
X_dendro = X_scaled[dendro_index]

Z_single = linkage(
    X_dendro,
    method="single",
)

fig, ax = plt.subplots(figsize=(11, 5))
dendrogram(
    Z_single,
    no_labels=True,
    ax=ax,
)
ax.set_xlabel("Customer leaves")
ax.set_ylabel("Merge distance")
ax.set_title("Single-linkage dendrogram")
plt.tight_layout()
plt.show()

<Figure size 1100x500 with 1 Axes>

In [22]:
Z_complete = linkage(
    X_dendro,
    method="complete",
)

fig, ax = plt.subplots(figsize=(11, 5))
dendrogram(
    Z_complete,
    no_labels=True,
    ax=ax,
)
ax.set_xlabel("Customer leaves")
ax.set_ylabel("Merge distance")
ax.set_title("Complete-linkage dendrogram")
plt.tight_layout()
plt.show()

<Figure size 1100x500 with 1 Axes>

In [23]:
Z_average = linkage(
    X_dendro,
    method="average",
)

fig, ax = plt.subplots(figsize=(11, 5))
dendrogram(
    Z_average,
    no_labels=True,
    ax=ax,
)
ax.set_xlabel("Customer leaves")
ax.set_ylabel("Merge distance")
ax.set_title("Average-linkage dendrogram")
plt.tight_layout()
plt.show()

<Figure size 1100x500 with 1 Axes>

In [24]:
Z_ward = linkage(
    X_dendro,
    method="ward",
)

fig, ax = plt.subplots(figsize=(11, 5))
dendrogram(
    Z_ward,
    no_labels=True,
    ax=ax,
)
ax.set_xlabel("Customer leaves")
ax.set_ylabel("Ward merge cost")
ax.set_title("Ward-linkage dendrogram")
plt.tight_layout()
plt.show()

<Figure size 1100x500 with 1 Axes>

The shapes differ noticeably. Single linkage creates long chains of merges at relatively low distances, which is consistent with its tendency to connect clusters through nearby points. Ward linkage delays larger merges until more distinct compact groups are combined.

The dendrogram also shows why HAC does not require a final cluster count before fitting. The hierarchy is built first; the analyst can then choose a cut height that retains a useful number of groups.

## 15. DBSCAN, HAC, and k-means answer different questions

These methods should not be expected to produce identical customer groups.

### K-means
Week 10 forced every customer into one of \(k\) centroid-based groups. It worked well for creating a complete customer segmentation.

### DBSCAN
DBSCAN found dense regions and explicitly left unusual observations as noise. That is useful when the goal includes anomaly or outlier detection rather than assigning everyone to a segment.

### HAC
HAC preserved a hierarchy. Its result depends strongly on the linkage rule, and the dendrogram lets the analyst inspect multiple possible cut levels instead of committing to one \(k\) at the start.

Ward linkage was the closest hierarchical analogue to the compact k-means structure, while single linkage was much more sensitive to chaining and isolated observations.

## 16. Expected and unexpected results

### What I expected

I expected DBSCAN to identify noise because Week 10 already showed that some customers have unusually high ordering, coupon, recency, or cashback behavior. I also expected Ward linkage to look more like k-means than single linkage because both favor compact clusters built around squared-error geometry.

### What surprised me

The strongest HAC Silhouette score came from single and average linkage at \(k=2\), but those solutions essentially isolated one customer from everyone else. The metric was high even though the segmentation was not operationally useful.

DBSCAN also produced a meaningful retention pattern without seeing churn: the complaint-heavy dense group had much higher observed churn than the main non-complaining group.

These results reinforced that clustering quality cannot be judged from one score alone. Cluster size, stability, noise rate, profile differences, and business interpretation all matter.

## 17. What this means for the customer-journey problem

Week 10 showed that k-means can create a complete set of customer segments. Week 11 adds two useful perspectives.

First, DBSCAN shows that some customers do not fit naturally into the dominant dense regions. Many of those noise customers have unusually high order, coupon, recency, or cashback behavior. Instead of forcing them into an average segment, they could be reviewed separately as unusual engagement patterns.

Second, both DBSCAN and Ward HAC again separate groups with materially different churn rates even though churn was not used to create the clusters. That supports the broader retention finding that complaint behavior, tenure, and recent transaction patterns define meaningful differences in the customer journey.

For an operational segmentation I would still prefer a method that assigns customers consistently and can be validated over time. For exploratory work, DBSCAN is especially useful for identifying unusual customers, while HAC is useful for understanding how broader and narrower customer groupings relate to one another.

## 18. Limitations

- DBSCAN is sensitive to \(\epsilon\), MinPts, and feature scaling.
- One global \(\epsilon\) can struggle when different parts of the data have very different densities.
- DBSCAN noise does not automatically mean a customer is bad or anomalous in a business sense.
- HAC results depend strongly on linkage choice.
- The HAC comparison uses a 1,200-customer sample because full hierarchical clustering is computationally heavier.
- Dendrograms use an even smaller sample for readability.
- Silhouette score can reward partitions that isolate a very small number of observations.
- PCA is used only for visualization and does not display the full seven-dimensional clustering geometry.
- Churn differences are post-hoc associations, not causal effects.

## 19. Reproducibility notes

- Random seed: 42
- Dataset: `data/churn_clean.csv`
- Clustering features: 7 numeric customer variables
- Scaling: StandardScaler fit on the clustering features
- Churn excluded from all clustering inputs
- DBSCAN final setting: \(\epsilon = 1.2\), MinPts = 14
- DBSCAN sensitivity: epsilon 0.5 to 2.2; MinPts 7, 10, 14, 20, 30
- HAC sample: 1,200 customers
- HAC linkage methods: single, complete, average, Ward
- HAC cluster counts evaluated: 2 through 6
- Dendrogram sample: 250 customers
- Week 10 k-means comparison: k = 4 on the same HAC sample
- Cluster comparison metric: Adjusted Rand Index
- Internal cluster metric: Silhouette score
- External profile variable: observed churn rate

## Reference for further reading

For implementation details and parameter definitions, I would use the scikit-learn clustering documentation together with the Week 11 course material. The course notes are particularly useful for the conceptual differences among DBSCAN point types and HAC linkage rules.